# WeightedKgBlend — Self-Contained Kaggle Notebook
**All pipeline code is embedded here. You only need to:**
1. Upload this notebook to Kaggle
2. Add the MIND dataset as input data
3. Set GPU accelerator: Settings → Accelerator → GPU T4 x2
4. Run All Cells


In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────
import subprocess, sys
pkgs = ['pykeen>=1.10.0','optuna>=3.0.0','scipy>=1.10.0','scikit-learn>=1.3.0']
for p in pkgs:
    subprocess.run([sys.executable,'-m','pip','install',p,'--quiet'], check=True)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory//1024**3} GB')
else:
    print('WARNING: No GPU — go to Settings -> Accelerator -> GPU T4 x2')


In [ ]:
# ── CELL 2: Setup paths + auto-download MIND from Zenodo ────────────────
from pathlib import Path
import shutil, os, urllib.request

WORK    = Path('/kaggle/working')
DATA    = WORK / 'data'
SPLITS  = DATA / 'splits'
RES     = WORK / 'results'
INPUT   = Path('/kaggle/input')

for d in [DATA, SPLITS, RES, WORK/'logs']:
    d.mkdir(parents=True, exist_ok=True)

MIND_PATH = None

# ── 1. Check if dataset already attached via Kaggle 'Add Data' ──────────────
tsvs = sorted(INPUT.glob('**/*.tsv'), key=lambda f: f.stat().st_size, reverse=True)
if tsvs:
    MIND_PATH = DATA / 'mind.tsv'
    shutil.copy(tsvs[0], MIND_PATH)
    print(f'Using attached dataset: {tsvs[0].name} ({tsvs[0].stat().st_size/1024**2:.0f} MB)')
else:
    # ── 2. Try Zenodo: download train.txt + test.txt + valid.txt and merge ───
    print('No attached dataset found. Downloading MIND from Zenodo...')
    ZENODO = 'https://zenodo.org/api/records/8117748/files'
    parts  = []
    for fname in ['train.txt', 'test.txt', 'valid.txt']:
        dest = DATA / fname
        if not dest.exists():
            url = f'{ZENODO}/{fname}/content'
            print(f'  Downloading {fname}...')
            urllib.request.urlretrieve(url, dest)
        sz = dest.stat().st_size / 1024**2
        print(f'  {fname}: {sz:.1f} MB')
        parts.append(dest)
    # Concatenate all three into a single mind.tsv
    MIND_PATH = DATA / 'mind.tsv'
    print('Merging into mind.tsv...')
    with open(MIND_PATH, 'wb') as out:
        for p in parts:
            with open(p, 'rb') as f:
                shutil.copyfileobj(f, out)
    print(f'mind.tsv ready: {MIND_PATH.stat().st_size/1024**2:.0f} MB')

# ── 3. Quick sanity check ───────────────────────────────────────────────────
if MIND_PATH and MIND_PATH.exists():
    import pandas as pd
    sample = pd.read_csv(MIND_PATH, sep='\t', header=None, names=['h','r','t'], nrows=50000)
    print(f'\nTriples in first 50k rows: {len(sample)}')
    print(f'Top relations:')
    print(sample['r'].value_counts().head(10))
else:
    raise RuntimeError('Could not locate or download MIND dataset!')


In [ ]:
# ── CELL 3: Set config ─────────────────────────────────────────────────────
# Adjust INDICATION_REL below to match your dataset's relation name for drug->disease
# Common values: 'indication', 'treats', 'drugind'
# The cell above prints top relations — check which one looks like drug->disease

INDICATION_REL = 'indication'   # <-- change if needed based on Cell 2 output
N_SPLITS       = 5
EMBEDDING_DIM  = 128            # reduced from 256 for Kaggle memory
NUM_EPOCHS     = 300
BATCH_SIZE     = 512
PATIENCE       = 15
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED    = 42

print(f'Indication relation : {INDICATION_REL}')
print(f'Splits              : {N_SPLITS}')
print(f'Embedding dim       : {EMBEDDING_DIM}')
print(f'Epochs              : {NUM_EPOCHS}')
print(f'Device              : {DEVICE}')


In [ ]:
# ── CELL 4: Prepare 5 random splits ───────────────────────────────────────
import pandas as pd
import numpy as np
import random

print('Loading MIND dataset...')
full = pd.read_csv(MIND_PATH, sep='\t', header=None, names=['head','relation','tail'])
print(f'Total triples: {len(full):,}')
print(f'Relations:     {full.relation.nunique()}')

ind = full[full.relation == INDICATION_REL].reset_index(drop=True)
print(f'Indication triples: {len(ind):,}')

if len(ind) == 0:
    print(f'ERROR: No triples found for relation "{INDICATION_REL}"')
    print('Check the top relations printed in Cell 2 and update INDICATION_REL in Cell 3')
else:
    non_ind = full[full.relation != INDICATION_REL]
    seeds   = [RANDOM_SEED + i*100 for i in range(N_SPLITS)]

    for i, seed in enumerate(seeds):
        sl = SPLITS / f'slice_{i}'
        sl.mkdir(exist_ok=True)
        idx = list(range(len(ind)))
        random.seed(seed)
        random.shuffle(idx)
        n      = len(idx)
        n_test = int(n * 0.10)
        n_val  = int(n * 0.10)
        tr = ind.iloc[idx[n_test+n_val:]]
        te = ind.iloc[idx[:n_test]]
        va = ind.iloc[idx[n_test:n_test+n_val]]
        kge_train = pd.concat([non_ind, tr], ignore_index=True)
        kge_train.to_csv(sl/'kge_train.tsv', sep='\t', index=False, header=False)
        tr.to_csv(sl/'ind_train.tsv', sep='\t', index=False, header=False)
        te.to_csv(sl/'ind_test.tsv',  sep='\t', index=False, header=False)
        va.to_csv(sl/'ind_valid.tsv', sep='\t', index=False, header=False)
        ents = pd.Series(pd.unique(full[['head','tail']].values.ravel()))
        ents.to_csv(sl/'entities.txt', index=False, header=False)
        print(f'  slice_{i}: train={len(tr)} test={len(te)} valid={len(va)}')

    print('Splits ready!')


In [ ]:
# ── CELL 5: Train KGE models ───────────────────────────────────────────────
# Longest step — ~6-8 hrs. Skips already-completed slice/model combos.
import torch, pandas as pd, numpy as np
from pathlib import Path
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

KGE_MODELS = ['TransE','RotatE','DistMult','ComplEx']

def get_preds(model, factory, relation, device):
    model.eval()
    rel_id = factory.relation_to_id.get(relation)
    if rel_id is None: return pd.DataFrame()
    triples = factory.mapped_triples
    mask    = triples[:,1] == rel_id
    rows    = []
    with torch.no_grad():
        for triple in triples[mask]:
            h,r,t = triple[0].item(), triple[1].item(), triple[2].item()
            h_t = torch.tensor([h], device=device).repeat(factory.num_entities)
            r_t = torch.tensor([r], device=device).repeat(factory.num_entities)
            scores = model.score_t(h_t, r_t)
            order  = torch.argsort(scores, descending=True).cpu().numpy()
            rank   = int(np.where(order == t)[0][0]) + 1
            rows.append({'drug': factory.entity_id_to_label[h],
                         'expected_disease': factory.entity_id_to_label[t],
                         'rank': rank, 'reciprocal_rank': 1.0/rank})
    return pd.DataFrame(rows)

for model_name in KGE_MODELS:
    for i in range(N_SPLITS):
        sl     = SPLITS / f'slice_{i}'
        outdir = RES / 'kge' / model_name / f'slice_{i}'
        done   = outdir / 'predictions_test.tsv'
        if done.exists():
            print(f'  SKIP {model_name}/slice_{i}')
            continue
        print(f'\n>>> {model_name} / slice_{i}...')
        outdir.mkdir(parents=True, exist_ok=True)
        tf_train = TriplesFactory.from_labeled_triples(
            pd.read_csv(sl/'kge_train.tsv', sep='\t', header=None,
                        names=['h','r','t']).values.astype(str))
        tf_test  = TriplesFactory.from_labeled_triples(
            pd.read_csv(sl/'ind_test.tsv', sep='\t', header=None,
                        names=['h','r','t']).values.astype(str),
            entity_to_id=tf_train.entity_to_id,
            relation_to_id=tf_train.relation_to_id)
        tf_valid = TriplesFactory.from_labeled_triples(
            pd.read_csv(sl/'ind_valid.tsv', sep='\t', header=None,
                        names=['h','r','t']).values.astype(str),
            entity_to_id=tf_train.entity_to_id,
            relation_to_id=tf_train.relation_to_id)
        res = pipeline(
            training=tf_train, testing=tf_test, validation=tf_valid,
            model=model_name,
            model_kwargs=dict(embedding_dim=EMBEDDING_DIM),
            optimizer='Adam', optimizer_kwargs=dict(lr=0.001),
            training_kwargs=dict(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE),
            stopper='early', stopper_kwargs=dict(patience=PATIENCE),
            device=DEVICE, random_seed=RANDOM_SEED)
        for split_name, tf in [('test',tf_test),('valid',tf_valid)]:
            preds = get_preds(res.model, tf, INDICATION_REL, DEVICE)
            preds.to_csv(outdir/f'predictions_{split_name}.tsv', sep='\t', index=False)
            print(f'   {split_name} MRR: {preds.reciprocal_rank.mean():.4f}')

print('\nKGE complete!')


In [ ]:
# ── CELL 6: CBR and ProbCBR ────────────────────────────────────────────────
import pandas as pd, numpy as np
from collections import defaultdict
from sklearn.cluster import MiniBatchKMeans

def build_graph(tsv):
    df = pd.read_csv(tsv, sep='\t', header=None, names=['h','r','t'])
    g  = defaultdict(set)
    for _, row in df.iterrows():
        g[row.h].add((row.r, row.t))
        g[row.t].add((f'inv_{row.r}', row.h))
    return dict(g)

def cbr_predict(drug, graph, train_triples, k, relation):
    q_edges = graph.get(drug, set())
    nb_scores = {}
    for h,r,t in train_triples:
        if h == drug: continue
        overlap = len(q_edges & graph.get(h, set()))
        if overlap: nb_scores[h] = nb_scores.get(h,0) + overlap
    top_nb = sorted(nb_scores, key=nb_scores.get, reverse=True)[:k]
    total  = sum(nb_scores.get(n,1) for n in top_nb) or 1
    scores = defaultdict(float)
    for nb in top_nb:
        w = nb_scores.get(nb,1)/total
        for h,r,t in train_triples:
            if h == nb and r == relation: scores[t] += w
    return dict(scores)

def cluster_entities(tsv, n_clusters):
    df   = pd.read_csv(tsv, sep='\t', header=None, names=['h','r','t'])
    od   = df.groupby('h').size().to_dict()
    ind_ = df.groupby('t').size().to_dict()
    ents = list(set(df.h.tolist()+df.t.tolist()))
    X    = np.array([[od.get(e,0), ind_.get(e,0)] for e in ents], dtype=np.float32)
    X   /= (X.max(axis=0)+1e-8)
    km   = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=5)
    return {e:int(l) for e,l in zip(ents, km.fit_predict(X))}

def run_method(name, sl, out_base, k=100, n_clusters=10):
    for split in ['test','valid']:
        outdir = out_base / name / sl.name
        outdir.mkdir(parents=True, exist_ok=True)
        done = outdir / f'predictions_{split}.tsv'
        if done.exists():
            print(f'  SKIP {name}/{sl.name}/{split}'); continue
        tr_triples = [(r.h,r.r,r.t) for r in
                      pd.read_csv(sl/'ind_train.tsv', sep='\t', header=None,
                                  names=['h','r','t']).itertuples()]
        ev_triples = [(r.h,r.r,r.t) for r in
                      pd.read_csv(sl/f'ind_{split}.tsv', sep='\t', header=None,
                                  names=['h','r','t']).itertuples()]
        graph = build_graph(sl/'kge_train.tsv')
        clusters = cluster_entities(sl/'kge_train.tsv', n_clusters) if name=='ProbCBR' else {}
        n_ents = len(pd.read_csv(sl/'entities.txt', header=None))
        rows = []
        for drug,rel,exp_dis in ev_triples:
            scores = cbr_predict(drug, graph, tr_triples, k, INDICATION_REL)
            if not scores: rank = n_ents
            else:
                sd = sorted(scores, key=scores.get, reverse=True)
                rank = sd.index(exp_dis)+1 if exp_dis in sd else n_ents
            rows.append({'drug':drug,'expected_disease':exp_dis,
                         'rank':rank,'reciprocal_rank':1.0/rank})
        df_out = pd.DataFrame(rows)
        df_out.to_csv(done, sep='\t', index=False)
        print(f'  {name}/{sl.name}/{split} MRR={df_out.reciprocal_rank.mean():.4f}')

for i in range(N_SPLITS):
    sl = SPLITS / f'slice_{i}'
    run_method('CBR',     sl, RES/'cbr')
    run_method('ProbCBR', sl, RES/'cbr')

print('CBR/ProbCBR complete!')


In [ ]:
# ── CELL 7: WeightedKgBlend ensemble (Optuna) ──────────────────────────────
import pandas as pd, numpy as np, optuna, yaml
optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL_PATHS = {
    'TransE':   RES/'kge'/'TransE',
    'RotatE':   RES/'kge'/'RotatE',
    'DistMult': RES/'kge'/'DistMult',
    'ComplEx':  RES/'kge'/'ComplEx',
    'CBR':      RES/'cbr'/'CBR',
    'ProbCBR':  RES/'cbr'/'ProbCBR',
}

def load_rr(base, sl, split):
    p = base / sl / f'predictions_{split}.tsv'
    if not p.exists(): return None
    df = pd.read_csv(p, sep='\t')
    rc = [c for c in df.columns if 'reciprocal' in c][0]
    return df[['drug','expected_disease',rc]].rename(columns={rc:'rr'})

for i in range(N_SPLITS):
    sl     = f'slice_{i}'
    outdir = RES/'ensemble'/sl
    outdir.mkdir(parents=True, exist_ok=True)
    if (outdir/'predictions_test.tsv').exists():
        print(f'SKIP ensemble/{sl}'); continue

    loaded, valid_dfs, test_dfs = [], [], []
    for mname, mpath in MODEL_PATHS.items():
        vdf = load_rr(mpath, sl, 'valid')
        tdf = load_rr(mpath, sl, 'test')
        if vdf is not None and tdf is not None:
            loaded.append(mname)
            valid_dfs.append(vdf); test_dfs.append(tdf)

    print(f'\nEnsemble {sl} — models: {loaded}')
    base_v = valid_dfs[0][['drug','expected_disease']].copy()
    base_t = test_dfs[0][['drug','expected_disease']].copy()
    for j,df in enumerate(valid_dfs):
        base_v = base_v.merge(df.rename(columns={'rr':f'rr_{j}'}),
                              on=['drug','expected_disease'], how='left')
        base_t = base_t.merge(df.rename(columns={'rr':f'rr_{j}'}),
                              on=['drug','expected_disease'], how='left')
    rcols  = [f'rr_{j}' for j in range(len(loaded))]
    base_v[rcols] = base_v[rcols].fillna(0)
    base_t[rcols] = base_t[rcols].fillna(0)
    Xv = base_v[rcols].values
    Xt = base_t[rcols].values

    def objective(trial):
        raw = np.array([trial.suggest_float(f'w{k}',0,1) for k in range(len(loaded))])
        if raw.sum() < 1e-8: return 0.0
        w = raw/raw.sum()
        return float(np.mean(Xv @ w))

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=200, show_progress_bar=True)
    raw_best = np.array([study.best_params[f'w{k}'] for k in range(len(loaded))])
    w_best   = raw_best / raw_best.sum()
    print(f'  Best valid MRR: {study.best_value:.4f}')
    print(f'  Weights: {dict(zip(loaded, w_best.round(3)))}')

    yaml.dump({'weights': dict(zip(loaded, w_best.round(4).tolist()))},
              open(outdir/'weights.yaml','w'))

    for split_name, X, base in [('test',Xt,base_t),('valid',Xv,base_v)]:
        out = base[['drug','expected_disease']].copy()
        out['ensemble_reciprocal_rank'] = X @ w_best
        out.to_csv(outdir/f'predictions_{split_name}.tsv', sep='\t', index=False)

print('Ensemble complete!')


In [ ]:
# ── CELL 8: Evaluate — mean±std across splits + Wilcoxon tests ────────────
import pandas as pd, numpy as np
from scipy.stats import wilcoxon

METRICS = ['MRR','Hits@1','Hits@3','Hits@10']
ALL_MODELS = list(MODEL_PATHS.keys()) + ['WeightedKgBlend']

def hits(rr, k): return float((rr >= 1/k).mean())
def metrics(rr): return {'MRR':float(rr.mean()),'Hits@1':hits(rr,1),
                         'Hits@3':hits(rr,3),'Hits@10':hits(rr,10)}

per_split = {m:{k:[] for k in METRICS} for m in ALL_MODELS}
raw_rr    = {m:[] for m in ALL_MODELS}

for i in range(N_SPLITS):
    sl = f'slice_{i}'
    for mname, mpath in MODEL_PATHS.items():
        p = mpath/sl/'predictions_test.tsv'
        if not p.exists(): [per_split[mname][k].append(np.nan) for k in METRICS]; continue
        df = pd.read_csv(p, sep='\t')
        rc = [c for c in df.columns if 'reciprocal' in c][0]
        rr = df[rc]
        m  = metrics(rr)
        for k in METRICS: per_split[mname][k].append(m[k])
        raw_rr[mname].extend(rr.tolist())

    ep = RES/'ensemble'/sl/'predictions_test.tsv'
    if ep.exists():
        df  = pd.read_csv(ep, sep='\t')
        rr  = df['ensemble_reciprocal_rank']
        m   = metrics(rr)
        for k in METRICS: per_split['WeightedKgBlend'][k].append(m[k])
        raw_rr['WeightedKgBlend'].extend(rr.tolist())
    else:
        [per_split['WeightedKgBlend'][k].append(np.nan) for k in METRICS]

# Summary table
rows = []
for m in ALL_MODELS:
    row = {'Algorithm': m}
    for k in METRICS:
        vals = [v for v in per_split[m][k] if not np.isnan(v)]
        row[k]         = f'{np.mean(vals):.4f} ± {np.std(vals):.4f}' if vals else 'N/A'
        row[f'{k}_mean']= np.mean(vals) if vals else np.nan
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(RES/'final_table.csv', index=False)
print('=== Results (mean ± std across 5 splits) ===')
print(summary[['Algorithm']+METRICS].to_string(index=False))

# Wilcoxon tests
ens_rr = pd.Series(raw_rr['WeightedKgBlend'])
stat_rows = []
for m in MODEL_PATHS:
    if not raw_rr[m]: continue
    mr = pd.Series(raw_rr[m])
    mn = min(len(ens_rr), len(mr))
    diff = ens_rr.values[:mn] - mr.values[:mn]
    if np.all(diff==0): stat, p = np.nan, np.nan
    else: stat, p = wilcoxon(diff, alternative='greater')
    stat_rows.append({'Model':m,'W_stat':round(stat,2) if not np.isnan(stat) else 'N/A',
                      'p_value':round(p,4) if not np.isnan(p) else 'N/A',
                      'significant(p<0.05)': p < 0.05 if not np.isnan(p) else False})
stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(RES/'stats.csv', index=False)
print('\n=== Wilcoxon tests (WeightedKgBlend vs each model) ===')
print(stats_df.to_string(index=False))


In [ ]:
# ── CELL 9: Package results for download ──────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/weightedkgblend_results', 'zip',
                    root_dir='/kaggle/working/results')
sz = Path('/kaggle/working/weightedkgblend_results.zip').stat().st_size/1024**2
print(f'Results zipped: weightedkgblend_results.zip ({sz:.1f} MB)')
print()
print('To download: click Output tab (bottom right) -> weightedkgblend_results.zip')
print()
print('Files inside:')
print('  final_table.csv   — Table 1 for manuscript (mean±std)')
print('  stats.csv         — Wilcoxon p-values')
print('  ensemble/*/weights.yaml — optimised lambda values')
